In [ ]:
import pandas as pd
import numpy as np

def engenharia_de_stints_treino(csv_entrada, csv_saida):
    print("Analisando telemetria de Treino: Caçando pits ocultos e montando stints")

    # 1. Carrega a base limpa do Treino Livre
    df = pd.read_csv(csv_entrada, sep=';', decimal=',')

    # 2. Extrai o número do carro (já que pulamos a etapa de fusão com o PDF de pits)
    df['Carro'] = df['Piloto'].str.extract(r'^(\d+)').astype(str)
    
    # Organiza os dados
    df['Lap_Int'] = df['Lap'].astype(int)
    df = df.sort_values(by=['Carro', 'Lap_Int']).reset_index(drop=True)

    # 3. O "Pit Stop Virtual"
    # Qualquer volta acima de 120 segundos (2 minutos) é considerada uma ida aos boxes/garagem
    df['Pit_Virtual'] = df['Lap Tm (Segundos)'] > 120

    # 4. Criar a coluna de Stint
    # O Stint muda na volta SEGUINTE à passagem pela garagem (na Out-Lap)
    df['Mudanca_Stint'] = df.groupby('Carro')['Pit_Virtual'].shift(1).fillna(False).astype(int)
    df['Stint'] = df.groupby('Carro')['Mudanca_Stint'].cumsum() + 1

    # 5. Classificar o Tipo de Volta
    df['Tipo_Volta'] = 'Push' 
    
    # Marca as voltas de garagem
    df.loc[df['Pit_Virtual'], 'Tipo_Volta'] = 'In-Lap/Garage'
    
    # Marca as Out-Laps
    out_lap_mask = df.groupby('Carro')['Pit_Virtual'].shift(1).fillna(False)
    df.loc[out_lap_mask, 'Tipo_Volta'] = 'Out-Lap'

    # 6. Detecção de Outliers (Tráfego ou voltas de resfriamento)
    df['Outlier'] = False
    df.loc[df['Tipo_Volta'] != 'Push', 'Outlier'] = True

    # Calcula a mediana do tempo de volta APENAS das voltas 'Push' do Stint
    df_push = df[df['Tipo_Volta'] == 'Push']
    medias_stint = df_push.groupby(['Carro', 'Stint'])['Lap Tm (Segundos)'].transform('median')
    
    df.loc[df['Tipo_Volta'] == 'Push', 'Tempo_Referencia_Stint'] = medias_stint
    
    # No Treino Livre, o tráfego é pior. Vamos usar um limite de 4% de tolerância
    limite_tempo = df['Tempo_Referencia_Stint'] * 1.04
    mascara_lenta = (df['Tipo_Volta'] == 'Push') & (df['Lap Tm (Segundos)'] > limite_tempo)
    df.loc[mascara_lenta, 'Outlier'] = True

    # Limpeza e reorganização
    df = df.drop(columns=['Lap_Int', 'Pit_Virtual', 'Mudanca_Stint', 'Tempo_Referencia_Stint'])
    
    # Move a coluna 'Carro' para o começo
    colunas = df.columns.tolist()
    colunas.insert(0, colunas.pop(colunas.index('Carro')))
    df = df[colunas]

    df.to_csv(csv_saida, index=False, sep=';', decimal=',')
    print(f"Base Estratégica de Treino concluída! Salva em: {csv_saida}")

# --- ÁREA DE EXECUÇÃO ---
arquivo_limpo_t2 = '../data/02_interim/dados_limpos_T2.csv'
arquivo_estrategia_t2 = '../data/03_processed/TELEMETRIA_ESTRATEGIA_T2.csv'

engenharia_de_stints_treino(arquivo_limpo_t2, arquivo_estrategia_t2)

🧠 Analisando telemetria de Treino: Caçando pits ocultos e montando stints...
📊 Base Estratégica de Treino concluída! Salva em: ../data/03_processed/TELEMETRIA_ESTRATEGIA_T2.csv
